# Lab 2 — Logistic regression embeddings and a neural classifier

COMP-672 · John Glossner

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glossner/COMP-672-Labs/blob/main/labs/week02_starter.ipynb)

**First: File → Save a copy in Drive.** Sign in to Google to run and save. Rename your copy, for example `COMP672_Lab02_YourName.ipynb`, and work in that copy. 

[Course lab index and submission instructions](https://github.com/glossner/COMP-672-Labs/blob/main/README.md) · [Handout](https://github.com/glossner/COMP-672-Labs/blob/main/labs/LAB02.md)


## Install the lab environment

Use Colab’s **default Python CPU runtime**; you do not need to select an older Python runtime. 


In [ ]:
import sys
import subprocess
import importlib.metadata as metadata

if not (3, 11) <= sys.version_info[:2] <= (3, 14):
    raise RuntimeError("This lab supports Python 3.11–3.14. Reopen the latest notebook from the course README if your runtime is newer.")

print("Runtime Python:", sys.version.split()[0])

LAB_PACKAGES = {'numpy': '2.1.3', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'matplotlib': '3.10.0'}
# Python 3.14 needs newer binary wheels; retain the established pins on 3.11–3.13.
if sys.version_info[:2] >= (3, 14):
    LAB_PACKAGES.update({'numpy': '2.3.5', 'scikit-learn': '1.7.2', 'matplotlib': '3.10.8'})
def installed_version(name):
    try:
        return metadata.version(name).split("+")[0]
    except metadata.PackageNotFoundError:
        return None

needed = [f"{name}=={version}" for name, version in LAB_PACKAGES.items()
          if name != "torch" and installed_version(name) != version]
if needed:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--only-binary=:all:", *needed])
print("Package setup complete. Restart the session if Colab requests it.")


In [ ]:
import platform
import importlib

LAB_VERSIONS = {"python": platform.python_version()}
for distribution, module_name in [("numpy", "numpy"), ("scipy", "scipy"),
                                  ("scikit-learn", "sklearn"), ("matplotlib", "matplotlib")]:
    module = importlib.import_module(module_name)
    LAB_VERSIONS[distribution] = module.__version__
for name, expected in LAB_PACKAGES.items():
    if LAB_VERSIONS[name].split("+")[0] != expected:
        raise RuntimeError(f"Restart the session: {name} is already imported at a different version.")
print(LAB_VERSIONS)
print("CPU environment ready. Synthetic data will be generated by the lab functions.")


# Lab 2 — Logistic regression embeddings and a neural classifier

Chapters 4, 5, 6

## Implementation tasks

1. Implement stable mean binary cross-entropy and its gradients with the specified L2 convention. Check both weights and bias by finite differences.

2. Implement batch gradient descent and report whether the training objective falls.

3. Implement cosine similarity with similarity defined as 0 when either vector has zero norm. Inspect five nearest document vectors under the SVD representation.

4. Implement a paired document bootstrap for the accuracy difference between two fixed predictions.


Functions to complete: `loss_gradient`, `train_logistic`, `cosine`, `paired_bootstrap`.

## Required experiments

- Run the fixed 2×2 comparison: TF-IDF or 16-dimensional SVD features crossed with linear or MLP classification. Report development and test macro F1 and the two-by-two confusion matrices.

- Analyze at least four negated examples. Add a bigram-feature comparison selected on development data and explain how feature expressiveness affects the linear baseline.

## Brief analysis in your notebook or code comments

1. Why is the generated label rule difficult for an additive unigram classifier?

2. What part of the comparison measures representation and what part measures nonlinearity?

3. What uncertainty does the paired interval capture, and what training variability does it omit?


Answer each question in approximately 2–4 sentences beside the results. This is part of the lab, with no separate report.



## Starter code

Complete the exercise functions below. Keep the supplied checks and add the cases required by the handout.

In [ ]:
"""Week 2 student starter: factorial feature/model comparison on synthetic text."""


In [ ]:
import numpy as np


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer


In [ ]:
from sklearn.decomposition import TruncatedSVD


In [ ]:
from sklearn.neural_network import MLPClassifier


In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix


In [ ]:
def sigmoid(z):
    z=np.asarray(z); return np.exp(-np.logaddexp(0,-z))


In [ ]:
def loss_gradient(X,y,w,b,l2=0.):
    """Mean binary NLL + l2/2 times squared weight norm."""
    raise NotImplementedError("Complete this laboratory task.")


In [ ]:
def train_logistic(X,y,steps=800,lr=.5,l2=.001):
    raise NotImplementedError("Complete this laboratory task.")


In [ ]:
def data():
    texts=[]; labels=[]
    for subject in ['sensor','radio','camera','screen','battery','motor','cable','speaker']:
        for adjective,positive in [('good',1),('reliable',1),('poor',0),('faulty',0)]:
            for negated in [False,True]:
                for place in ['today','indoors','outside','again','overnight']:
                    texts.append(f'the {subject} was {"not " if negated else ""}{adjective} {place}')
                    labels.append(int(bool(positive)!=negated))
    ids=np.arange(len(texts)); y=np.array(labels)
    tr,other=train_test_split(ids,test_size=.4,stratify=y,random_state=17)
    dev,te=train_test_split(other,test_size=.5,stratify=y[other],random_state=18)
    return np.array(texts),y,tr,dev,te


In [ ]:
def cosine(a,b):
    """Return cosine similarity; define it as 0 if either vector has zero norm."""
    raise NotImplementedError("Complete this laboratory task.")


In [ ]:
def paired_bootstrap(y,a,b,iterations=500,seed=17):
    """CI for accuracy(a)-accuracy(b), paired by document."""
    raise NotImplementedError("Complete this laboratory task.")


In [ ]:
def checks():
    X=np.array([[2.,-1.],[0.,1.]]); y=np.array([1.,0.]); w=np.array([.3,-.2]); b=.1
    _,g,_=loss_gradient(X,y,w,b)
    for i in range(2):
        d=np.zeros(2); d[i]=1e-5
        numeric=(loss_gradient(X,y,w+d,b)[0]-loss_gradient(X,y,w-d,b)[0])/2e-5
        assert np.isclose(g[i],numeric,atol=1e-6)
    assert np.isclose(cosine(np.array([1,1,0]),np.array([1,0,1])),.5)


In [ ]:
def experiment():
    texts,y,tr,dev,te=data()
    vectorizer=TfidfVectorizer(); raw=vectorizer.fit_transform(texts[tr])
    X={name:vectorizer.transform(texts[ix]).toarray() for name,ix in [('train',tr),('dev',dev),('test',te)]}
    svd=TruncatedSVD(n_components=16,random_state=17).fit(raw)
    dense={name:svd.transform(value) for name,value in X.items()}
    results={}; predictions={}
    for feature,views in [('tfidf',X),('svd16',dense)]:
        for model in ['linear','mlp']:
            if model=='linear':
                w,b=train_logistic(views['train'],y[tr]); predict=lambda z:(sigmoid(z@w+b)>=.5).astype(int)
            else:
                fit=MLPClassifier(hidden_layer_sizes=(24,),max_iter=1000,random_state=17,alpha=.001,learning_rate_init=.01).fit(views['train'],y[tr]); predict=fit.predict
            a=predict(views['dev']); p=predict(views['test']); key=feature+'_'+model
            results[key]={'dev_macro_f1':f1_score(y[dev],a,average='macro'),'test_macro_f1':f1_score(y[te],p,average='macro'),'test_accuracy':accuracy_score(y[te],p),'confusion':confusion_matrix(y[te],p).tolist()}
            predictions[key]=p
    results['paired_accuracy_ci_tfidf_mlp_minus_linear']=paired_bootstrap(y[te],predictions['tfidf_mlp'],predictions['tfidf_linear'])
    results['note']='SVD produces corpus-derived dense vectors; these are not pretrained word2vec embeddings.'
    return results


## Run checks and the supplied experiment

Complete the exercise functions first. A `NotImplementedError` here is expected in an untouched starter. The result dictionary covers the supplied baseline; the handout also requires comparisons and analysis that you must add below.


In [ ]:
import time

checks()
_started = time.perf_counter()
results = experiment()
baseline_runtime_seconds = time.perf_counter() - _started
results


## Your required comparisons

Add the additional experiments, checks, plots/tables, measured runtimes, and clearly labeled results required by the handout. Add their JSON-compatible summaries to `additional_results`.


In [ ]:
# TODO: implement and run the remaining required comparisons here.
additional_results = {}


## Brief analysis

1. Why is the generated label rule difficult for an additive unigram classifier?

2. What part of the comparison measures representation and what part measures nonlinearity?

3. What uncertainty does the paired interval capture, and what training variability does it omit?

Write approximately 2–4 sentences for each question in the next cell, tied to your measured results.

1. _Your response:_

2. _Your response:_

3. _Your response:_


## Save and submit

From Colab: **Share → General access → Anyone with the link → Viewer**. Click **Copy link**. Paste the link into the Canvas assignment submission.
